# DermaScan — EfficientNet-B3 retraining (Kaggle GPU → CPU deployment)

Trains the skin-lesion classifier on HAM10000 and exports a **self-describing,
CPU-ready** artifact bundle that drops straight into the serving repository.

## What this notebook fixes

Each of these caused a real defect in the deployed system. They are designed in
here rather than patched afterwards.

| Past failure | How this notebook prevents it |
| :--- | :--- |
| Served checkpoint silently differed from the evaluated one (20-point accuracy gap) | The checkpoint carries its own `arch`, `head`, `classes` and a weight fingerprint. Serving verifies rather than assumes. |
| Inference ran at 224 px centre-crop while training used 300 px | `img_size` and the exact normalization are stored **in** the checkpoint; validation transforms are identical to serving transforms. |
| Thresholds fitted on softmax, served under sigmoid | `readout` is recorded and the thresholds are fitted under that same readout. |
| Confidence ~0.97 on wrong answers (ECE 0.116) | Temperature fitted on a held-out calibration split and exported. |
| Melanoma recall 0.624, misses reported as benign | Melanoma-weighted loss, plus an alert threshold fitted for a target sensitivity. |
| OOD gate biased against darker skin | Brightness/contrast augmentation, and an explicit brightness-robustness check before export. |
| `torch.load` failed on CPU due to numpy scalars in the checkpoint | Everything saved is a tensor or a plain Python primitive, so `weights_only=True` works. |
| Metrics reported that were never measured | The test split is touched exactly once, at the end, and never used for any fitting. |
| A checkpoint from an earlier session was silently restored and evaluated (run 2: val 0.81, test 0.05) | Checkpoints carry a `RUN_ID`; the restore asserts it, rebuilds the model from scratch, and re-measures validation before anything downstream runs. |
| The melanoma floor turned the decision rule into "predict melanoma always" (run 2: 97.5% of cases flagged) | The floor is a penalty, not a flat infeasible region, so the search can find its way into the feasible set instead of falling back to a degenerate rule. |
| Temperature and thresholds exported as a pair that was never evaluated together | Each `(T, thresholds)` pair is scored as a pair and the best jointly-measured one is shipped. |

## Before running

1. **Settings → Accelerator → GPU** (P100 or T4).
2. **Add data**: search for `skin-cancer-mnist-ham10000` and add it.
3. **Save Version -> Save & Run All (Commit)**, not an interactive Run All.
   Kaggle deletes `/kaggle/working` when an unsaved interactive session ends,
   and closing the browser tab is enough to end one. Run 3 was lost that way:
   a model that passed the release gate, with no way to recover the weights.
   A commit run executes top to bottom on Kaggle's side and keeps its output
   as a permanent version. Roughly 1 h at 35 epochs on a T4.
4. Optional: upload `models/latest.pt` as a dataset to run the incumbent
   comparison cell.

In [2]:
# ── Environment ──────────────────────────────────────────────────────────────
import copy, os, sys, json, math, random, time
from collections import Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu  ", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Settings -> Accelerator -> GPU.")

try:
    import timm
except ImportError:
    !pip install -q timm
    import timm
print("timm ", timm.__version__)

torch 2.10.0+cu128 | cuda True
gpu   Tesla T4
timm  1.0.26


In [3]:
# ── Configuration ────────────────────────────────────────────────────────────
CFG = {
    "seed": 42,

    # Architecture. "plain" = timm head (matches the current serving default);
    # "multihead" = the two-layer head. Whichever is chosen is recorded in the
    # checkpoint, so serving can never guess wrong again.
    "arch": "efficientnet_b3",
    "head": "plain",
    "num_classes": 7,

    # Must match serving exactly. Stored in the checkpoint.
    "img_size": 300,
    "norm_mean": [0.485, 0.456, 0.406],
    "norm_std": [0.229, 0.224, 0.225],

    # Readout the thresholds will be fitted under, and that serving must use.
    "readout": "softmax",

    "epochs": 35,
    "batch_size": 32,

    # Run 2 fine-tuned the backbone at 3e-5 and plateaued at val macro-F1 0.677
    # by epoch 16 while train loss kept falling to 0.27 - the head was learning
    # and the backbone was barely moving. More backbone movement, plus mixup as
    # the regulariser that pays for it, is the trade being made here.
    "lr_head": 5e-4,
    "lr_backbone": 1e-4,
    "weight_decay": 1e-4,
    "warmup_epochs": 2,
    "label_smoothing": 0.05,
    "focal_gamma": 2.0,
    "drop_rate": 0.3,
    "drop_path_rate": 0.2,

    # Mixup: soft targets across a shuffled batch. It is the cheapest defence
    # against that epoch-16 plateau and it improves calibration, which matters
    # directly here because the thresholds are fitted on probabilities.
    "mixup_alpha": 0.2,
    "mixup_prob": 0.5,

    # Exponential moving average of the weights. Evaluated and checkpointed
    # INSTEAD of the raw weights, and it is the EMA that gets exported.
    "ema_decay": 0.9995,

    # Melanoma is the class whose errors matter most. This multiplies its loss
    # contribution on top of inverse-frequency weighting.
    "mel_loss_weight": 2.5,
    "target_mel_sensitivity": 0.90,

    # Sampler strength. 1.0 = fully class-balanced batches, 0.0 = natural
    # frequencies. 0.5 (square root) is the usual compromise; full balancing
    # plus class weights is a double correction that collapsed the first run.
    # 0.6 leans a little further towards the rare classes, which is where the
    # macro-F1 is actually being lost - df has 60 training images.
    "sampler_power": 0.6,

    # Threshold fitting must not trade melanoma away: the first run maximised
    # macro-F1 alone and pushed the melanoma threshold to 0.75, cutting melanoma
    # recall from 0.76 (argmax) to 0.57.
    "min_mel_recall": 0.70,

    "early_stop_patience": 10,
    "num_workers": 2,
    "amp": True,

    # ── Release gate ─────────────────────────────────────────────────────────
    # Absolute criteria, measured on the lesion-disjoint test split. These are
    # the deploy decision, in place of "beat the incumbent", because the
    # incumbent's published numbers were measured on an image-level split.
    # HAM10000 holds ~10015 images of ~7470 lesions, so an image-level split
    # puts other photographs of the SAME lesion in both train and test, and the
    # resulting accuracy is optimistic by an unknown margin. Comparing an honest
    # number against an optimistic one is not a decision procedure. Cell "fair
    # comparison" below re-measures the incumbent on THIS split when the file is
    # available; until it has, treat the published numbers as unusable.
    "release_gate": {
        "macro_f1": 0.70,           # at least
        "melanoma_recall": 0.70,    # at least
        "melanoma_surfaced": 0.90,  # at least, with the alert channel
        "review_rate": 0.45,        # at most
        "ece": 0.10,                # at most
    },
}

CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]   # ORDER IS CONTRACT
MEL_IDX = CLASSES.index("mel")
OUT_DIR = "/kaggle/working"

# Every checkpoint this run writes is tagged with RUN_ID. /kaggle/working is
# repopulated from the previous version whenever a notebook is re-opened, so a
# weights file left by an earlier run is always sitting there waiting to be
# loaded by accident. Run 2 restored one and calibrated it: test accuracy came
# out at 0.053 - below the 0.143 you would get by guessing - moments after
# training had reported val accuracy 0.81.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False   # speed; determinism is seeded above
    torch.backends.cudnn.benchmark = True

seed_everything(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("run id:", RUN_ID)
print(json.dumps(CFG, indent=2))

run id: 20260903-043324
{
  "seed": 42,
  "arch": "efficientnet_b3",
  "head": "plain",
  "num_classes": 7,
  "img_size": 300,
  "norm_mean": [
    0.485,
    0.456,
    0.406
  ],
  "norm_std": [
    0.229,
    0.224,
    0.225
  ],
  "readout": "softmax",
  "epochs": 35,
  "batch_size": 32,
  "lr_head": 0.0005,
  "lr_backbone": 0.0001,
  "weight_decay": 0.0001,
  "warmup_epochs": 2,
  "label_smoothing": 0.05,
  "focal_gamma": 2.0,
  "drop_rate": 0.3,
  "drop_path_rate": 0.2,
  "mixup_alpha": 0.2,
  "mixup_prob": 0.5,
  "ema_decay": 0.9995,
  "mel_loss_weight": 2.5,
  "target_mel_sensitivity": 0.9,
  "sampler_power": 0.6,
  "min_mel_recall": 0.7,
  "early_stop_patience": 10,
  "num_workers": 2,
  "amp": true,
  "release_gate": {
    "macro_f1": 0.7,
    "melanoma_recall": 0.7,
    "melanoma_surfaced": 0.9,
    "review_rate": 0.45,
    "ece": 0.1
  }
}


## 1. Data

HAM10000 contains multiple images of the *same lesion*. Splitting by image leaks
the same lesion into train and test and inflates every metric. Splits below are
**grouped by `lesion_id`** and stratified by diagnosis.

In [4]:
import os
import pandas as pd

# ── 1. Locate the dataset dynamically ────────────────────────────────────────
ROOT = None

# Walk through every single directory inside /kaggle/input
for dirpath, _, files in os.walk('/kaggle/input'):
    # Convert file names to lowercase to catch any casing variations
    if any("ham10000_metadata" in f.lower() for f in files):
        ROOT = dirpath
        break

if ROOT is None:
    # If still not found, print exactly what Kaggle sees to debug
    contents = os.listdir('/kaggle/input/') if os.path.exists('/kaggle/input/') else 'No input folder'
    raise SystemExit(f"Data not found. /kaggle/input/ contents: {contents}. Try re-adding the dataset.")

print("dataset root:", ROOT)

# ── 2. Load Metadata ─────────────────────────────────────────────────────────
meta_path = None
for f in os.listdir(ROOT):
    if f.lower().startswith("ham10000_metadata"):
        meta_path = os.path.join(ROOT, f)
        break

meta = pd.read_csv(meta_path)
print("metadata rows:", len(meta))

# ── 3. Map Image Paths ───────────────────────────────────────────────────────
index = {}

# Search from the ROOT down to find all .jpg files (handles part_1/part_2 splits)
for dirpath, _, files in os.walk(ROOT):
    for f in files:
        if f.lower().endswith(".jpg"):
            index[os.path.splitext(f)[0]] = os.path.join(dirpath, f)

meta["path"] = meta["image_id"].map(index)
missing = meta["path"].isna().sum()
meta = meta.dropna(subset=["path"]).reset_index(drop=True)

print(f"images located: {len(meta)} | missing: {missing}")
print(meta["dx"].value_counts().to_string())

dataset root: /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000
metadata rows: 10015
images located: 10015 | missing: 0
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115


In [5]:
# ── Lesion-disjoint stratified splits ────────────────────────────────────────
# One lesion contributes all of its images to exactly one split.
def make_splits(df, seed, fractions=(0.60, 0.15, 0.10, 0.15)):
    rng = np.random.RandomState(seed)
    lesion_dx = df.groupby("lesion_id")["dx"].first()
    assign = {}
    names = ["train", "val", "calib", "test"]
    for dx, group in lesion_dx.groupby(lesion_dx):
        lesions = group.index.to_numpy()
        rng.shuffle(lesions)
        cuts = (np.cumsum(fractions) * len(lesions)).astype(int)
        for name, chunk in zip(names, np.split(lesions, cuts[:-1])):
            for les in chunk:
                assign[les] = name
    out = df.copy()
    out["split"] = out["lesion_id"].map(assign)
    return out

data = make_splits(meta, CFG["seed"])
print(pd.crosstab(data["split"], data["dx"], margins=True).to_string())

# Leakage assertions - cheap, and this is exactly the class of bug that silently
# inflates results.
for a in ["train", "val", "calib", "test"]:
    for b in ["train", "val", "calib", "test"]:
        if a >= b: continue
        overlap = set(data[data.split == a].lesion_id) & set(data[data.split == b].lesion_id)
        assert not overlap, f"lesion leakage between {a} and {b}: {len(overlap)}"
assert data["split"].notna().all()
print("\nno lesion appears in more than one split")

for name in ["train", "val", "calib", "test"]:
    data[data.split == name][["image_id", "lesion_id", "dx", "path"]].to_csv(
        f"{OUT_DIR}/split_{name}.csv", index=False)
print("split manifests written")

dx     akiec  bcc   bkl   df   mel    nv  vasc    All
split                                                
calib     34   49   109   15   110   678    14   1009
test      52   79   158   22   170  1000    22   1503
train    196  310   669   60   663  3992    85   5975
val       45   76   163   18   170  1035    21   1528
All      327  514  1099  115  1113  6705   142  10015

no lesion appears in more than one split
split manifests written


In [6]:
# ── Transforms ───────────────────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image

NORM = transforms.Normalize(mean=CFG["norm_mean"], std=CFG["norm_std"])
S = CFG["img_size"]

# Brightness and contrast jitter is not cosmetic: the deployed system was found
# to change its answer when an image was darkened, which biases against darker
# skin and poor lighting. Training across that range is the fix.
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(S, scale=(0.75, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomApply([transforms.RandomRotation(25)], p=0.5),
    transforms.ColorJitter(brightness=0.30, contrast=0.30,
                           saturation=0.20, hue=0.02),
    transforms.ToTensor(),
    NORM,
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10)),
])

# IDENTICAL to serving. Any divergence here is the 224-vs-300 bug all over again.
eval_tf = transforms.Compose([
    transforms.Resize((S, S)),
    transforms.ToTensor(),
    NORM,
])

class LesionDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = [CLASSES.index(d) for d in frame["dx"]]
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]

def loader_for(split, transform, shuffle=False, sampler=None):
    ds = LesionDataset(data[data.split == split], transform)
    return DataLoader(ds, batch_size=CFG["batch_size"], shuffle=shuffle,
                      sampler=sampler, num_workers=CFG["num_workers"],
                      pin_memory=True, drop_last=(sampler is not None))

# Sampling: nv is ~67% of the corpus and dominates the gradient, but FULL
# inverse-frequency sampling combined with inverse-frequency loss weights is a
# double correction (~1/freq^2). The first run of this notebook did exactly that
# and collapsed: validation accuracy was 0.08 at epoch 1, when predicting only nv
# would have scored 0.68. Square-root balancing evens the classes out without
# pretending the real distribution is uniform.
train_frame = data[data.split == "train"]
counts = Counter(train_frame["dx"])
weights = train_frame["dx"].map(lambda d: (1.0 / counts[d]) ** CFG["sampler_power"]).to_numpy()
sampler = WeightedRandomSampler(torch.DoubleTensor(weights), len(weights),
                                replacement=True)
print("sampler power", CFG["sampler_power"], "(1.0 = fully balanced, 0.0 = natural)")

train_loader = loader_for("train", train_tf, sampler=sampler)
val_loader   = loader_for("val", eval_tf)
calib_loader = loader_for("calib", eval_tf)
test_loader  = loader_for("test", eval_tf)
print({k: len(data[data.split == k]) for k in ["train", "val", "calib", "test"]})

sampler power 0.6 (1.0 = fully balanced, 0.0 = natural)
{'train': 5975, 'val': 1528, 'calib': 1009, 'test': 1503}


## 2. Model and loss

The head choice is recorded in the checkpoint. `plain` matches the serving
default; `multihead` reproduces the two-layer head.

In [7]:
# ── Model ────────────────────────────────────────────────────────────────────
class ClassifierHead(nn.Module):
    '''Two-layer head with independent dropout gates.'''
    def __init__(self, in_features, num_classes, hidden=512, drop=0.5):
        super().__init__()
        mid = hidden // 2
        self.head = nn.Sequential(
            nn.Linear(in_features, hidden), nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True), nn.Dropout(drop),
            nn.Linear(hidden, mid), nn.BatchNorm1d(mid),
            nn.ReLU(inplace=True), nn.Dropout(drop),
            nn.Linear(mid, num_classes))
    def forward(self, x):
        return self.head(x)

class MultiHeadNet(nn.Module):
    def __init__(self, arch, num_classes, drop, drop_path):
        super().__init__()
        self.backbone = timm.create_model(arch, pretrained=True, num_classes=0,
                                          drop_rate=drop, drop_path_rate=drop_path)
        self.classifier = ClassifierHead(self.backbone.num_features, num_classes,
                                         drop=drop)
    def forward(self, x):
        f = self.backbone.forward_features(x)
        if f.dim() == 4: f = f.mean(dim=[2, 3])
        elif f.dim() == 3: f = f.mean(dim=1)
        return self.classifier(f)

def build_model():
    if CFG["head"] == "plain":
        return timm.create_model(CFG["arch"], pretrained=True,
                                 num_classes=CFG["num_classes"],
                                 drop_rate=CFG["drop_rate"],
                                 drop_path_rate=CFG["drop_path_rate"])
    if CFG["head"] == "multihead":
        return MultiHeadNet(CFG["arch"], CFG["num_classes"],
                            CFG["drop_rate"], CFG["drop_path_rate"])
    raise ValueError("unknown head: " + CFG["head"])

model = build_model().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"{CFG['arch']} / {CFG['head']} head — {n_params/1e6:.1f}M parameters")

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

efficientnet_b3 / plain head — 10.7M parameters


In [8]:
# ── Loss ─────────────────────────────────────────────────────────────────────
# Focal loss over softmax, with an explicit extra multiplier on melanoma. The
# deployed model missed ~38% of melanomas and reported them as benign; weighting
# that error is the main lever available without collecting more data.
# The sampler already handles frequency. Applying inverse-frequency weights here
# as well double-corrects, and in the first run it left melanoma weighted 0.66 -
# BELOW akiec (0.89) and df (2.92) - so the melanoma multiplier never did its
# job. Weights are uniform; only melanoma is boosted, which is the whole intent.
class_weights = np.ones(len(CLASSES), dtype=np.float64)
class_weights[MEL_IDX] = CFG["mel_loss_weight"]
print("class weights:", {c: round(float(w), 3) for c, w in zip(CLASSES, class_weights)})
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

class FocalLoss(nn.Module):
    """Focal loss over a soft target distribution.

    Taking a distribution rather than an index is what lets mixup share this
    loss: a mixed batch is just q = lam*q_a + (1-lam)*q_b. Given a smoothed
    one-hot q it is identical to the index-based version it replaces.
    """
    def __init__(self, weight, gamma, smoothing):
        super().__init__()
        self.weight, self.gamma, self.smoothing = weight, gamma, smoothing

    def soft_target(self, target, n):
        q = torch.zeros(target.size(0), n, device=target.device,
                        dtype=torch.float32).fill_(self.smoothing / (n - 1))
        return q.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)

    def forward_soft(self, logits, q):
        logp = F.log_softmax(logits.float(), dim=1)
        ce = -(q * logp).sum(dim=1)
        pt = (q * logp.exp()).sum(dim=1)          # confidence in the target mix
        w = (q * self.weight.unsqueeze(0)).sum(dim=1)
        return (w * (1 - pt).pow(self.gamma) * ce).mean()

    def forward(self, logits, target):
        with torch.no_grad():
            q = self.soft_target(target, logits.size(1))
        return self.forward_soft(logits, q)

criterion = FocalLoss(class_weights_t, CFG["focal_gamma"], CFG["label_smoothing"])

def mixup(x, y, alpha):
    """Returns the mixed images, the permuted labels, and the mixing weight."""
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1.0 - lam) * x[idx], y[idx], lam

# ── Weight EMA ───────────────────────────────────────────────────────────────
class ModelEma:
    """Exponential moving average of the weights, evaluated in place of the raw
    model. It costs one extra copy of a 10.7M-parameter net and it removes most
    of the epoch-to-epoch thrash: in Run 2 melanoma recall swung between 0.59
    and 0.82 on consecutive epochs, which makes checkpoint selection a lottery.
    """
    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay
        self.updates = 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        # Ramp the decay in, or the average spends its first epochs anchored to
        # the random initialisation.
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            src = msd[k].detach()
            if v.dtype.is_floating_point:
                v.mul_(d).add_(src, alpha=1.0 - d)
            else:
                v.copy_(src)          # BN num_batches_tracked and friends

ema = ModelEma(model, CFG["ema_decay"])

# ── Differential learning rates ──────────────────────────────────────────────
# Split by module identity, not by substring. The previous version matched the
# name fragment "head", which silently swept efficientnet's conv_head - a
# backbone block of ~1.5M parameters - into the head group at 10x the rate it
# was meant to get.
head_module = model.get_classifier() if CFG["head"] == "plain" else model.classifier
head_ids = {id(p) for p in head_module.parameters()}
head_params = [p for p in model.parameters() if id(p) in head_ids]
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
assert head_params, "no head parameters found - check CFG['head']"
print(f"head {sum(p.numel() for p in head_params)/1e3:.0f}k @ lr {CFG['lr_head']}"
      f"  |  backbone {sum(p.numel() for p in backbone_params)/1e6:.1f}M"
      f" @ lr {CFG['lr_backbone']}")

optimizer = torch.optim.AdamW(
    [{"params": backbone_params, "lr": CFG["lr_backbone"]},
     {"params": head_params, "lr": CFG["lr_head"]}],
    weight_decay=CFG["weight_decay"])

steps = max(1, len(train_loader))
def lr_lambda(step):
    e = step / steps
    if e < CFG["warmup_epochs"]:
        return (e + 1e-8) / CFG["warmup_epochs"]
    p = (e - CFG["warmup_epochs"]) / max(1e-8, CFG["epochs"] - CFG["warmup_epochs"])
    return 0.5 * (1 + math.cos(math.pi * min(1.0, p)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=CFG["amp"] and DEVICE.type == "cuda")
print("optimizer, schedule, EMA ready")

class weights: {'akiec': 1.0, 'bcc': 1.0, 'bkl': 1.0, 'df': 1.0, 'mel': 2.5, 'nv': 1.0, 'vasc': 1.0}
head 11k @ lr 0.0005  |  backbone 10.7M @ lr 0.0001
optimizer, schedule, EMA ready


## 3. Training

Selection metric is macro-F1, with melanoma recall logged every epoch so a model
that trades melanoma away for overall accuracy is visible while it happens.

In [9]:
# ── Metric helpers ───────────────────────────────────────────────────────────
def per_class_scores(y_true, y_pred, n=len(CLASSES)):
    f1, rec, prec = [], [], []
    for i in range(n):
        tp = int(((y_pred == i) & (y_true == i)).sum())
        fp = int(((y_pred == i) & (y_true != i)).sum())
        fn = int(((y_pred != i) & (y_true == i)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        prec.append(p); rec.append(r)
        f1.append(2 * p * r / (p + r) if p + r else 0.0)
    return {"accuracy": float((y_pred == y_true).mean()),
            "macro_f1": float(np.mean(f1)),
            "precision": prec, "recall": rec, "f1": f1}

@torch.no_grad()
def collect_logits(net, loader):
    net.eval()
    outs, ys = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", enabled=CFG["amp"] and DEVICE.type == "cuda"):
            outs.append(net(x).float().cpu())
        ys.append(y)
    return torch.cat(outs).numpy(), torch.cat(ys).numpy()

In [10]:
# ── Train ────────────────────────────────────────────────────────────────────
# The checkpoint file name carries RUN_ID, and the file carries RUN_ID inside
# it. Nothing downstream is allowed to load weights it cannot prove came from
# this run - see the verification cell immediately below, which is the check
# Run 2 did not have.
BEST_PATH = f"{OUT_DIR}/_best_{RUN_ID}.pt"
FLOOR = CFG["min_mel_recall"]

def selection_score(m):
    """Macro-F1, with a linear penalty for melanoma recall below the floor.

    A hard constraint is wrong here: melanoma recall is measured on 170 val
    images and swings several points on noise alone, so a floor would throw away
    good epochs. A penalty keeps the pressure on without the cliff."""
    return m["macro_f1"] - 0.5 * max(0.0, FLOOR - m["recall"][MEL_IDX])

best = {"score": -1e9, "epoch": -1}
history = []
patience = 0

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    running, seen, t0, skipped = 0.0, 0, time.time(), 0
    for x, y in train_loader:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        use_mix = CFG["mixup_alpha"] > 0 and random.random() < CFG["mixup_prob"]
        with torch.autocast("cuda", enabled=CFG["amp"] and DEVICE.type == "cuda"):
            if use_mix:
                xm, y2, lam = mixup(x, y, CFG["mixup_alpha"])
                n = CFG["num_classes"]
                q = (lam * criterion.soft_target(y, n)
                     + (1.0 - lam) * criterion.soft_target(y2, n))
                loss = criterion.forward_soft(model(xm), q)
            else:
                loss = criterion(model(x), y)

        scaler.scale(loss).backward()
        # AMP skips the optimizer step on any batch whose gradients overflowed.
        # Stepping the schedule anyway advances the learning rate for an update
        # that never happened - which is what PyTorch's "lr_scheduler.step()
        # before optimizer.step()" warning was pointing at in Run 2.
        prev_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if scaler.get_scale() >= prev_scale:
            scheduler.step()
        else:
            skipped += 1
        ema.update(model)
        running += loss.item() * x.size(0); seen += x.size(0)

    # The EMA weights are the product of this run: evaluate those, not the raw
    # ones, so that what is selected is what is exported.
    logits, y_true = collect_logits(ema.module, val_loader)
    m = per_class_scores(y_true, logits.argmax(1))
    score = selection_score(m)
    history.append({"epoch": epoch, "train_loss": running / max(1, seen),
                    "val_accuracy": m["accuracy"], "val_macro_f1": m["macro_f1"],
                    "val_mel_recall": m["recall"][MEL_IDX],
                    "selection_score": score})
    print(f"epoch {epoch:3d}  loss {running/max(1,seen):.4f}  "
          f"val acc {m['accuracy']:.4f}  macroF1 {m['macro_f1']:.4f}  "
          f"melRecall {m['recall'][MEL_IDX]:.4f}  "
          f"({time.time()-t0:.0f}s"
          + (f", {skipped} amp skips" if skipped else "") + ")")

    if score > best["score"]:
        best = {"score": score, "epoch": epoch, "macro_f1": m["macro_f1"],
                "accuracy": m["accuracy"], "mel_recall": m["recall"][MEL_IDX]}
        torch.save({"run_id": RUN_ID,
                    "epoch": epoch,
                    "val_macro_f1": float(m["macro_f1"]),
                    "val_accuracy": float(m["accuracy"]),
                    "state_dict": {k: v.detach().cpu().clone()
                                   for k, v in ema.module.state_dict().items()}},
                   BEST_PATH)
        patience = 0
        print("            ^ best so far, checkpointed")
    else:
        patience += 1
        if patience >= CFG["early_stop_patience"]:
            print(f"early stopping: no improvement for {patience} epochs")
            break

pd.DataFrame(history).to_csv(f"{OUT_DIR}/training_history.csv", index=False)
print(f"\nbest epoch {best['epoch']}: val macro-F1 {best['macro_f1']:.4f}, "
      f"acc {best['accuracy']:.4f}, mel recall {best['mel_recall']:.4f}")

epoch   1  loss 3.2905  val acc 0.5308  macroF1 0.2710  melRecall 0.4588  (173s, 5 amp skips)
            ^ best so far, checkpointed
epoch   2  loss 1.4742  val acc 0.6774  macroF1 0.4519  melRecall 0.5412  (95s)
            ^ best so far, checkpointed
epoch   3  loss 0.9107  val acc 0.7225  macroF1 0.5642  melRecall 0.6235  (94s)
            ^ best so far, checkpointed
epoch   4  loss 0.7179  val acc 0.7480  macroF1 0.6148  melRecall 0.6059  (92s)
            ^ best so far, checkpointed
epoch   5  loss 0.6021  val acc 0.7605  macroF1 0.6157  melRecall 0.6294  (90s)
            ^ best so far, checkpointed
epoch   6  loss 0.5627  val acc 0.7814  macroF1 0.6435  melRecall 0.6176  (90s)
            ^ best so far, checkpointed
epoch   7  loss 0.4692  val acc 0.7925  macroF1 0.6516  melRecall 0.6412  (89s)
            ^ best so far, checkpointed
epoch   8  loss 0.4352  val acc 0.7965  macroF1 0.6596  melRecall 0.5882  (90s)
epoch   9  loss 0.3642  val acc 0.7997  macroF1 0.6621  melRecall 

In [11]:
# ── Restore the best weights, and PROVE they are the right ones ──────────────
# This cell exists because Run 2 did not have it. Training reported val accuracy
# 0.81; three cells later the same kernel measured 0.053 on test - below the
# 0.143 that guessing would give - because the weights that came back off disk
# were not the weights that had just been evaluated. Nothing distinguishes a
# stale checkpoint from a good one at load time except a check like this one.
#
# `model` is rebuilt from scratch first: after this cell, no result depends on
# any object left over in the kernel.
model = build_model().to(DEVICE)

ckpt = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
assert ckpt["run_id"] == RUN_ID, (
    f"checkpoint is from run {ckpt['run_id']}, this run is {RUN_ID}. "
    "A file from an earlier session was about to be evaluated - stop here.")
model.load_state_dict(ckpt["state_dict"])   # strict
model.eval()

v_logits, v_y = collect_logits(model, val_loader)
v = per_class_scores(v_y, v_logits.argmax(1))
expected = ckpt["val_macro_f1"]
drift = abs(v["macro_f1"] - expected)

print(f"restored epoch {ckpt['epoch']} of run {ckpt['run_id']}")
print(f"val macro-F1  logged {expected:.4f}   re-measured {v['macro_f1']:.4f}"
      f"   drift {drift:.4f}")
print(f"val accuracy  {v['accuracy']:.4f}   val mel recall {v['recall'][MEL_IDX]:.4f}")

if drift > 0.01:
    raise RuntimeError(
        f"restored weights do not reproduce the logged validation score "
        f"({expected:.4f} logged vs {v['macro_f1']:.4f} re-measured). Do not "
        f"calibrate or export this checkpoint - every number downstream would "
        f"describe a model you did not train.")
print("\nrestore verified - the calibration and test cells below are measuring "
      "the checkpoint that was actually selected")

restored epoch 19 of run 20260903-043324
val macro-F1  logged 0.7012   re-measured 0.7012   drift 0.0000
val accuracy  0.8312   val mel recall 0.6353

restore verified - the calibration and test cells below are measuring the checkpoint that was actually selected


In [12]:
# ── Training curves (real ones, from the logged history) ─────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(h.epoch, h.train_loss); ax[0].set_title("training loss")
ax[1].plot(h.epoch, h.val_macro_f1); ax[1].set_title("validation macro-F1")
ax[2].plot(h.epoch, h.val_mel_recall, color="#a8442a")
ax[2].set_title("validation melanoma recall")
for a in ax: a.set_xlabel("epoch"); a.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/training_curves.png", dpi=150)
plt.close(fig)
print("training_curves.png written from measured history — keep this file; the "
      "previous project had no training log and its curves could not be reproduced")

training_curves.png written from measured history — keep this file; the previous project had no training log and its curves could not be reproduced


## 4. Calibration and thresholds — fitted on `calib`, never on `test`

Three things are fitted here, all on the calibration split:

1. **Temperature** — flattens over-confident probabilities.
2. **Per-class thresholds** — the decision rule serving uses.
3. **Melanoma alert threshold** — flags `p(mel)` above a cutoff regardless of
   which class wins, because melanoma misses still carry substantial `p(mel)`.
Temperature and thresholds are fitted alternately, but only pairs that were *scored together* are eligible for export - see the notes in the cell below for what shipping an unevaluated pair cost run 2.

In [13]:
# ── Fit on the calibration split ─────────────────────────────────────────────
def to_probs(logits, temperature=1.0):
    z = torch.tensor(logits, dtype=torch.float32) / temperature
    if CFG["readout"] == "softmax":
        return torch.softmax(z, dim=1).numpy()
    return torch.sigmoid(z).numpy()

def decide(probs, thresholds):
    return np.argmax(probs - thresholds[None, :], axis=1)

def ece(conf, correct, bins=15):
    e = 0.0
    for i in range(bins):
        m = (conf > i / bins) & (conf <= (i + 1) / bins)
        if m.sum():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

cal_logits, cal_y = collect_logits(model, calib_loader)

# Temperature and thresholds interact: the confidence the server reports is the
# probability of the class the THRESHOLD RULE picked, not the arg-max. The first
# run fitted T on arg-max and then reported ECE on thresholded predictions, which
# is why calib ECE read 0.037 while test ECE read 0.237. Both are fitted here on
# the same path, alternating until they settle.

def decided_confidence(probs, thresholds):
    yp = decide(probs, thresholds)
    return yp, probs[np.arange(len(yp)), yp]

# The melanoma floor is a PENALTY, not a hard constraint. Run 2 encoded it as
# `if recall < floor: return -1`, which makes the objective flat at -1 across
# every infeasible threshold vector - coordinate ascent has no direction to move
# in and reports its own starting point as the answer. The recovery path then
# maximised `mel_recall + 0.001*macro_f1`, whose global optimum is "predict
# melanoma for every image": calib recall 1.0000, macro-F1 0.0300, and 97.5% of
# all test cases flagged for review. A model that flags everything has told you
# nothing.
#
# A linear penalty is feasibility-seeking where the constraint is violated and
# plain macro-F1 where it is met, so the search walks into the feasible region
# and then optimises inside it.
MEL_PENALTY = 5.0

def fit_thresholds(logits, y, temperature):
    """Maximise macro-F1 under a penalised melanoma-recall floor."""
    probs = to_probs(logits, temperature)
    floor = CFG["min_mel_recall"]

    def objective(tt):
        s = per_class_scores(y, decide(probs, tt))
        deficit = max(0.0, floor - s["recall"][MEL_IDX])
        return s["macro_f1"] - MEL_PENALTY * deficit

    def ascend(start):
        best_t = start.copy()
        best_v = objective(best_t)
        for _ in range(6):
            improved = False
            for i in range(len(CLASSES)):
                for cand in np.round(np.arange(0.0, 0.96, 0.05), 2):
                    trial = best_t.copy(); trial[i] = cand
                    v = objective(trial)
                    if v > best_v + 1e-9:
                        best_v, best_t, improved = v, trial, True
            if not improved:
                break
        return best_t, best_v

    # Several starts, because coordinate ascent is local: the arg-max rule, a
    # neutral rule, and one that opens melanoma up from the outset.
    mel_open = np.full(len(CLASSES), 0.5); mel_open[MEL_IDX] = 0.0
    best_t, best_v = None, -1e9
    for start in [np.full(len(CLASSES), 0.5), np.zeros(len(CLASSES)), mel_open]:
        t, v = ascend(start)
        if v > best_v:
            best_t, best_v = t, v

    s = per_class_scores(y, decide(probs, best_t))
    got = s["recall"][MEL_IDX]
    if got + 1e-9 < floor:
        # Say so plainly rather than degenerating into an all-melanoma rule.
        print(f"  WARNING: melanoma recall {got:.4f} is below the floor "
              f"{floor:.2f} and the penalised search could not reach it.")
        print(f"           The model does not separate melanoma well enough at "
              f"this operating point. Retrain - do not lower the floor to make "
              f"this line disappear.")
    return best_t, s["macro_f1"], got

def fit_temperature(logits, y, thresholds):
    best_t, best_e = 1.0, float("inf")
    for T in np.arange(0.5, 6.01, 0.05):
        probs = to_probs(logits, T)
        yp, conf = decided_confidence(probs, thresholds)
        e = ece(conf, yp == y)
        if e < best_e:
            best_t, best_e = float(T), e
    return best_t, best_e

# Alternating fits do not converge here, and the previous version shipped
# whatever the last iteration happened to leave behind: thresholds fitted under
# the PREVIOUS temperature, paired with a temperature fitted under the PREVIOUS
# thresholds. The pair it exported had never been evaluated together. On a
# synthetic replica of this problem the temperature oscillated 1.0 -> 6.0 -> 0.5
# and the exported pair scored macro-F1 0.699 / melanoma recall 0.573, while a
# consistent pair from the same data reached 0.781 / 0.700.
#
# So: keep every (temperature, thresholds) pair that was actually fitted
# together, score each one as a pair, and ship the best.
def score_pair(T, th):
    probs = to_probs(cal_logits, T)
    yp = decide(probs, th)
    s = per_class_scores(cal_y, yp)
    conf = probs[np.arange(len(yp)), yp]
    return {"T": float(T), "thresholds": th, "macro_f1": s["macro_f1"],
            "accuracy": s["accuracy"], "mel_recall": s["recall"][MEL_IDX],
            "ece": ece(conf, yp == cal_y),
            "mel_share": float((yp == MEL_IDX).mean())}

pairs, T = [], 1.0
for round_ in range(4):
    th, _, _ = fit_thresholds(cal_logits, cal_y, T)
    p = score_pair(T, th)                       # scored as the pair it is
    pairs.append(p)
    print(f"round {round_+1}: T={p['T']:.2f}  macro-F1={p['macro_f1']:.4f}  "
          f"melRecall={p['mel_recall']:.4f}  ECE={p['ece']:.4f}")
    T, _ = fit_temperature(cal_logits, cal_y, th)
    if any(abs(T - q["T"]) < 1e-9 for q in pairs):
        break                                   # temperature has started cycling

floor = CFG["min_mel_recall"]
feasible = [p for p in pairs if p["mel_recall"] + 1e-9 >= floor]
if feasible:
    # Among pairs that clear the melanoma floor: best macro-F1, with calibration
    # error as the tie-break.
    chosen = max(feasible, key=lambda p: p["macro_f1"] - 0.5 * p["ece"])
else:
    chosen = max(pairs, key=lambda p: p["macro_f1"] - MEL_PENALTY * (floor - p["mel_recall"]))
    print(f"\nWARNING: no fitted pair reached melanoma recall {floor:.2f}.")

best_T, thresholds, best_e = chosen["T"], chosen["thresholds"], chosen["ece"]
print(f"\nchosen pair: T={best_T:.2f}  (fitted and scored together)")

if best_T >= 5.95 or best_T <= 0.55:
    print("NOTE: temperature sits at the edge of the search grid. Check the "
          "macro-F1 above before trusting the reported confidences.")

print("\nthresholds       ", {c: round(float(t), 2) for c, t in zip(CLASSES, thresholds)})
cal_probs = to_probs(cal_logits, best_T)
cal_pred = decide(cal_probs, thresholds)
cal_scores = per_class_scores(cal_y, cal_pred)
print(f"calib macro-F1   {cal_scores['macro_f1']:.4f}")
print(f"calib accuracy   {cal_scores['accuracy']:.4f}")
print(f"calib mel recall {cal_scores['recall'][MEL_IDX]:.4f}  (floor {CFG['min_mel_recall']})")

# Melanoma alert threshold: lowest review rate reaching the target sensitivity.
mel_mask = cal_y == MEL_IDX
alert_t, alert_rate = None, 1.01
for t in np.round(np.arange(0.05, 0.96, 0.05), 2):
    surfaced = (cal_probs[:, MEL_IDX] >= t) | (cal_pred == MEL_IDX)
    if surfaced[mel_mask].mean() >= CFG["target_mel_sensitivity"]:
        rate = surfaced[~mel_mask].mean()
        if rate < alert_rate:
            alert_t, alert_rate = float(t), float(rate)
if alert_t is None:
    alert_t = 0.50
    surfaced = (cal_probs[:, MEL_IDX] >= alert_t) | (cal_pred == MEL_IDX)
    alert_rate = float(surfaced[~mel_mask].mean())
    print(f"WARNING: target melanoma sensitivity {CFG['target_mel_sensitivity']} is "
          f"unreachable on calib (best {surfaced[mel_mask].mean():.4f}); using 0.50")
print(f"melanoma alert   p(mel) >= {alert_t:.2f}  (calib review rate {alert_rate:.3f})")
if alert_rate > 0.5:
    print("                 a review rate above 0.5 means the alert channel is "
          "flagging most of the corpus and carries no information")

round 1: T=1.00  macro-F1=0.7219  melRecall=0.7000  ECE=0.0468
round 2: T=1.30  macro-F1=0.7133  melRecall=0.7000  ECE=0.0317

chosen pair: T=1.00  (fitted and scored together)

thresholds        {'akiec': 0.4, 'bcc': 0.75, 'bkl': 0.3, 'df': 0.3, 'mel': 0.05, 'nv': 0.7, 'vasc': 0.55}
calib macro-F1   0.7219
calib accuracy   0.7512
calib mel recall 0.7000  (floor 0.7)
melanoma alert   p(mel) >= 0.50  (calib review rate 0.199)


## 5. Final evaluation — the test split, used exactly once

Nothing below feeds back into any fitting decision. If these numbers are
disappointing, the honest response is to change the training recipe and retrain,
not to tune against this split.

In [14]:
# ── Test-set evaluation ──────────────────────────────────────────────────────
test_logits, test_y = collect_logits(model, test_loader)
test_probs = to_probs(test_logits, best_T)
test_pred = decide(test_probs, thresholds)

m = per_class_scores(test_y, test_pred)
conf = test_probs[np.arange(len(test_pred)), test_pred]
mel_mask_t = test_y == MEL_IDX
surfaced = (test_probs[:, MEL_IDX] >= alert_t) | (test_pred == MEL_IDX)

print(f"test images            {len(test_y)}")
print(f"accuracy               {m['accuracy']:.4f}")
print(f"macro-F1               {m['macro_f1']:.4f}")
print(f"ECE                    {ece(conf, test_pred == test_y):.4f}")
print(f"melanoma recall        {m['recall'][MEL_IDX]:.4f}")
print(f"melanoma surfaced      {surfaced[mel_mask_t].mean():.4f}  (with alert channel)")
print(f"cases flagged          {surfaced[~mel_mask_t].mean():.4f}")
print()
print(f"{'class':<8}{'support':>9}{'prec':>9}{'recall':>9}{'f1':>9}")
for i, c in enumerate(CLASSES):
    print(f"{c:<8}{int((test_y==i).sum()):>9}{m['precision'][i]:>9.4f}"
          f"{m['recall'][i]:>9.4f}{m['f1'][i]:>9.4f}")

# Both decision paths, because the first run of this notebook lost melanoma
# recall purely in the threshold step (0.76 at arg-max -> 0.57 thresholded).
argmax_pred = test_probs.argmax(1)
am = per_class_scores(test_y, argmax_pred)
print()
print(f"{'':<22}{'arg-max':>10}{'thresholded':>14}")
print(f"{'accuracy':<22}{am['accuracy']:>10.4f}{m['accuracy']:>14.4f}")
print(f"{'macro-F1':<22}{am['macro_f1']:>10.4f}{m['macro_f1']:>14.4f}")
print(f"{'melanoma recall':<22}{am['recall'][MEL_IDX]:>10.4f}{m['recall'][MEL_IDX]:>14.4f}")

# A sanity floor that costs nothing and would have caught Run 2 outright.
if am["accuracy"] < 1.0 / len(CLASSES):
    raise RuntimeError(
        f"arg-max accuracy {am['accuracy']:.4f} is below chance ({1/len(CLASSES):.3f}). "
        "The thresholds cannot cause this - arg-max ignores them. Something is "
        "wrong with the weights or the data path, not the calibration.")

candidate = {"accuracy": m["accuracy"], "macro_f1": m["macro_f1"],
             "melanoma_recall": m["recall"][MEL_IDX],
             "melanoma_surfaced": float(surfaced[mel_mask_t].mean()),
             "review_rate": float(surfaced[~mel_mask_t].mean()),
             "ece": ece(conf, test_pred == test_y)}

# ── Release gate: absolute criteria on the lesion-disjoint split ─────────────
LOWER_IS_BETTER = {"ece", "review_rate"}
print()
print(f"{'':<22}{'required':>11}{'measured':>11}{'':>4}")
failures = []
for k, req in CFG["release_gate"].items():
    got = candidate[k]
    ok = got <= req if k in LOWER_IS_BETTER else got >= req
    rel = "<=" if k in LOWER_IS_BETTER else ">="
    if not ok:
        failures.append(k)
    print(f"{k:<22}{rel + f'{req:.4f}':>11}{got:>11.4f}   {'pass' if ok else 'FAIL'}")

DEPLOYABLE = not failures
print()
if DEPLOYABLE:
    print("release gate: PASS")
else:
    print("release gate: FAIL on " + ", ".join(failures))
    print("Change the recipe and retrain. Tuning against these test numbers, or")
    print("loosening the gate to match them, would only overfit the split.")

# ── The published incumbent numbers, for reference only ─────────────────────
# These come from docs/evaluation_results.json: n=1525, image-level split, no
# lesion-disjointness. HAM10000 has ~10015 images of ~7470 lesions, so an
# image-level split puts other photographs of the same lesion on both sides and
# the numbers are optimistic by an unknown margin. They are printed because they
# are what is currently claimed in the repo, NOT because losing to them means
# this model is worse. Run the fair-comparison cell below for a real answer.
PUBLISHED_INCUMBENT = {"accuracy": 0.8505, "macro_f1": 0.7450,
                       "melanoma_recall": 0.6236, "melanoma_surfaced": 0.8989,
                       "review_rate": 0.3051, "ece": 0.0511}
print()
print(f"{'':<22}{'published*':>11}{'candidate':>11}")
for k, base in PUBLISHED_INCUMBENT.items():
    print(f"{k:<22}{base:>11.4f}{candidate[k]:>11.4f}")
print("* image-level split, not lesion-disjoint: optimistic, not comparable.")

cm = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
for t, p in zip(test_y, test_pred):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(8, 7))
norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(CLASSES)), CLASSES); ax.set_yticks(range(len(CLASSES)), CLASSES)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title(f"Measured confusion matrix (n={len(test_y)})")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, f"{norm[i,j]:.2f}\n({cm[i,j]})", ha="center", va="center",
                fontsize=7, color="white" if norm[i, j] > 0.5 else "black")
fig.colorbar(im, ax=ax); fig.tight_layout()
fig.savefig(f"{OUT_DIR}/confusion_matrix_measured.png", dpi=150); plt.close(fig)
print("\nconfusion_matrix_measured.png written")

test images            1503
accuracy               0.7771
macro-F1               0.6904
ECE                    0.0426
melanoma recall        0.8529
melanoma surfaced      0.8529  (with alert channel)
cases flagged          0.1673

class     support     prec   recall       f1
akiec          52   0.5536   0.5962   0.5741
bcc            79   0.9423   0.6203   0.7481
bkl           158   0.7241   0.5316   0.6131
df             22   0.6250   0.6818   0.6522
mel           170   0.3940   0.8529   0.5390
nv           1000   0.9517   0.8280   0.8856
vasc           22   0.9412   0.7273   0.8205

                         arg-max   thresholded
accuracy                  0.8244        0.7771
macro-F1                  0.7121        0.6904
melanoma recall           0.6471        0.8529

                         required   measured    
macro_f1                 >=0.7000     0.6904   FAIL
melanoma_recall          >=0.7000     0.8529   pass
melanoma_surfaced        >=0.9000     0.8529   FAIL
review_rate   

### 5b. The deployed model, re-measured on this split

The incumbent was trained on an unrecorded split of the same corpus, so most of the images called "test" here were in its training set. The cell below measures it anyway and then checks whether it scored better than it did on its own held-out data - which is the signature of exactly that contamination.

In [15]:
# ── The incumbent, re-measured on THIS split - and why that is not enough ────
# The repo publishes accuracy 0.8505 / macro-F1 0.7450 for the deployed model,
# measured on an image-level split of HAM10000. That dataset has ~10015 images of
# ~7470 lesions, so an image-level split routinely places two photographs of the
# same lesion on opposite sides of the train/test line, and the published number
# is optimistic by an unknown margin.
#
# Re-running the incumbent on the lesion-disjoint split below does NOT fix that,
# and it is important to understand why before reading the table. The split here
# is drawn fresh from all 10015 images. The incumbent was trained on its own
# split of the same corpus, which is unknown and unrecorded - so most of what is
# called "test" here was in that model's TRAINING set. It is being asked to
# recognise images it memorised.
#
# The tell is that the incumbent scores HIGHER here than on its own held-out
# data. When that happens the comparison is contaminated and must not be used to
# accept or reject anything; the check below states so explicitly rather than
# printing a table that looks authoritative.
INCUMBENT_PATHS = [f"{OUT_DIR}/latest.pt", "/kaggle/input/dermascan-incumbent/latest.pt"]
incumbent_file = next((p for p in INCUMBENT_PATHS if os.path.exists(p)), None)

if incumbent_file is None:
    for dirpath, _, files in os.walk("/kaggle/input"):
        if "latest.pt" in files:
            incumbent_file = os.path.join(dirpath, "latest.pt")
            break

incumbent_fair = None
if incumbent_file is None:
    print("incumbent checkpoint not found - skipping.")
    print("Note that finding it would not settle the comparison either; see the")
    print("contamination note above.")
else:
    print("incumbent:", incumbent_file)
    blob = torch.load(incumbent_file, map_location="cpu", weights_only=False)
    state = blob.get("model_state_dict", blob.get("state_dict", blob))
    state = {k.replace("_orig_mod.", "").replace("module.", ""): v
             for k, v in state.items()}
    inc = timm.create_model(CFG["arch"], pretrained=False,
                            num_classes=CFG["num_classes"])
    missing, unexpected = inc.load_state_dict(state, strict=False)
    if missing or unexpected:
        print(f"  NOTE: {len(missing)} missing / {len(unexpected)} unexpected keys "
              f"under a plain {CFG['arch']} head - if those are not zero the "
              f"incumbent has a different architecture and nothing below is valid.")
    inc = inc.to(DEVICE).eval()

    inc_logits, inc_y = collect_logits(inc, test_loader)
    inc_m = per_class_scores(inc_y, inc_logits.argmax(1))
    incumbent_fair = {"accuracy": inc_m["accuracy"], "macro_f1": inc_m["macro_f1"],
                      "melanoma_recall": inc_m["recall"][MEL_IDX]}

    print()
    print(f"{'arg-max, this split':<22}{'incumbent':>11}{'candidate':>11}")
    for k, label, mine in [
            ("accuracy", "accuracy", am["accuracy"]),
            ("macro_f1", "macro-F1", am["macro_f1"]),
            ("melanoma_recall", "melanoma recall", am["recall"][MEL_IDX])]:
        print(f"{label:<22}{incumbent_fair[k]:>11.4f}{mine:>11.4f}")

    # What the incumbent itself reported on data it held out.
    own = blob.get("metrics", {})
    own_acc = own.get("accuracy")
    print()
    if own_acc:
        print(f"the incumbent's own held-out accuracy, from its checkpoint: {own_acc:.4f}")
    if own_acc and incumbent_fair["accuracy"] > own_acc + 0.02:
        # Solve  measured = seen_frac * ~1.0 + (1 - seen_frac) * own_acc  for a
        # rough sense of how much of this split the incumbent has already seen.
        seen = (incumbent_fair["accuracy"] - own_acc) / max(1e-6, 1.0 - own_acc)
        print()
        print("!" * 70)
        print(f"CONTAMINATED: the incumbent scores {incumbent_fair['accuracy']:.4f} here but")
        print(f"{own_acc:.4f} on its own held-out data. A model does not do BETTER on a")
        print("stricter split. Roughly " + f"{seen:.0%}" + " of these images were in its training")
        print("set, so the column above measures memorisation, not skill.")
        print()
        print("This comparison cannot be repaired without the incumbent's original")
        print("split manifest, which was never recorded. Judge the candidate on the")
        print("release gate above - absolute criteria on data it has never seen -")
        print("and treat the incumbent's numbers as unverifiable.")
        print("!" * 70)
    else:
        print("no contamination signal; the table above is a like-for-like arg-max")
        print("comparison on data neither model was fitted on.")

    del inc
    torch.cuda.empty_cache()

incumbent: /kaggle/input/models/uehskufhkjdshoihb/latest/pytorch/default/1/latest.pt

arg-max, this split     incumbent  candidate
accuracy                   0.9534     0.8244
macro-F1                   0.9288     0.7121
melanoma recall            0.8000     0.6471

the incumbent's own held-out accuracy, from its checkpoint: 0.8637

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
CONTAMINATED: the incumbent scores 0.9534 here but
0.8637 on its own held-out data. A model does not do BETTER on a
stricter split. Roughly 66% of these images were in its training
set, so the column above measures memorisation, not skill.

This comparison cannot be repaired without the incumbent's original
split manifest, which was never recorded. Judge the candidate on the
release gate above - absolute criteria on data it has never seen -
and treat the incumbent's numbers as unverifiable.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


In [16]:
# ── Brightness robustness ────────────────────────────────────────────────────
# The deployed model changed its answer when an image was darkened. That biases
# against darker skin tones and poor lighting. This measures whether training
# augmentation actually fixed it. Darkening is a crude proxy for melanin, not a
# substitute for evaluating on a dataset with Fitzpatrick labels (e.g. DDI).
_MEAN = torch.tensor(CFG["norm_mean"]).view(1, 3, 1, 1)
_STD = torch.tensor(CFG["norm_std"]).view(1, 3, 1, 1)

@torch.no_grad()
def accuracy_at_brightness(factor):
    '''Scale brightness in pixel space, not in normalized space.'''
    model.eval()
    correct = total = 0
    for x, y in test_loader:
        pixels = (x * _STD + _MEAN).clamp(0, 1) * factor      # de-normalise, dim
        x = ((pixels.clamp(0, 1) - _MEAN) / _STD).to(DEVICE)  # re-normalise
        with torch.autocast("cuda", enabled=CFG["amp"] and DEVICE.type == "cuda"):
            p = model(x).float().cpu()
        correct += int((p.argmax(1) == y).sum()); total += len(y)
    return correct / total

print("brightness scale -> accuracy")
robust = {}
for f in [1.0, 0.75, 0.5, 0.35]:
    robust[f] = accuracy_at_brightness(f)
    print(f"  x{f:<5} {robust[f]:.4f}")
spread = max(robust.values()) - min(robust.values())
print(f"\nspread {spread:.4f}  ({'acceptable' if spread < 0.05 else 'STILL BRIGHTNESS-SENSITIVE'})")

brightness scale -> accuracy
  x1.0   0.8244
  x0.75  0.8263
  x0.5   0.8150
  x0.35  0.8051

spread 0.0213  (acceptable)


## 6. Export — CPU-ready, self-describing

The bundle stores everything serving needs to reproduce training conditions, so
the architecture/resolution/readout mismatches that broke the previous
deployment cannot recur. Only tensors and plain primitives are saved, so
`torch.load(..., weights_only=True)` works on a CPU-only machine.

In [17]:
# ── Export ───────────────────────────────────────────────────────────────────
import hashlib

# CPU, float32, no torch.compile / DataParallel prefixes, no numpy scalars.
state = {}
for k, v in model.state_dict().items():
    k = k.replace("_orig_mod.", "").replace("module.", "")
    state[k] = v.detach().cpu().float()

fingerprint = hashlib.sha256(
    b"".join(state[k].numpy().tobytes() for k in sorted(state))).hexdigest()[:16]

def py(x):
    '''numpy scalars break torch.load(weights_only=True); force plain floats.'''
    return float(x)

test_metrics = {
    "test_set_size": int(len(test_y)),
    "accuracy": py(m["accuracy"]),
    "macro_f1": py(m["macro_f1"]),
    "ece": py(ece(conf, test_pred == test_y)),
    "melanoma_recall": py(m["recall"][MEL_IDX]),
    "melanoma_surfaced": py(surfaced[mel_mask_t].mean()),
    "review_rate": py(surfaced[~mel_mask_t].mean()),
    "per_class": {c: {"support": int((test_y == i).sum()),
                      "precision": py(m["precision"][i]),
                      "recall": py(m["recall"][i]),
                      "f1": py(m["f1"][i])}
                  for i, c in enumerate(CLASSES)},
    "confusion_matrix": cm.tolist(),
}

bundle = {
    "format_version": 1,
    "run_id": RUN_ID,
    "created": datetime.now(timezone.utc).isoformat(),
    "arch": CFG["arch"],
    "head": CFG["head"],
    "num_classes": CFG["num_classes"],
    "classes": CLASSES,
    "img_size": CFG["img_size"],
    "norm_mean": CFG["norm_mean"],
    "norm_std": CFG["norm_std"],
    "readout": CFG["readout"],
    "decision_rule": "argmax(probability - class_threshold)",
    "temperature": py(best_T),
    "thresholds": {c: py(t) for c, t in zip(CLASSES, thresholds)},
    "mel_alert_threshold": py(alert_t),
    "weight_fingerprint": fingerprint,
    "best_epoch": int(best["epoch"]),
    "val_macro_f1": py(best["macro_f1"]),
    "weights": "EMA",
    "model_state_dict": state,
}
torch.save(bundle, f"{OUT_DIR}/dermascan_b3.pt")

# Backend-compatible sidecars, so this drops into the serving repo unchanged.
with open(f"{OUT_DIR}/class_thresholds.json", "w") as f:
    json.dump({"class_thresholds": {c: py(t) for c, t in zip(CLASSES, thresholds)},
               "readout": CFG["readout"],
               "fitted_on": "calibration split (lesion-disjoint)",
               "per_class_metrics": {
                   c: {"threshold": py(thresholds[i]),
                       "f1": py(m["f1"][i]), "precision": py(m["precision"][i]),
                       "recall": py(m["recall"][i])}
                   for i, c in enumerate(CLASSES)}}, f, indent=2)

with open(f"{OUT_DIR}/calibration.json", "w") as f:
    json.dump({"temperature": py(best_T), "mel_alert_threshold": py(alert_t),
               "readout": CFG["readout"],
               "target_sensitivity": CFG["target_mel_sensitivity"],
               "fitted_on": "calibration split (lesion-disjoint)",
               "reported_on": "held-out test split",
               "test_metrics_after": {
                   "accuracy": test_metrics["accuracy"],
                   "ece": test_metrics["ece"],
                   "melanoma_recall_argmax": test_metrics["melanoma_recall"],
                   "melanoma_surfaced": test_metrics["melanoma_surfaced"],
                   "review_rate": test_metrics["review_rate"]}}, f, indent=2)

with open(f"{OUT_DIR}/evaluation_results.json", "w") as f:
    json.dump(test_metrics, f, indent=2)
with open(f"{OUT_DIR}/train_config.json", "w") as f:
    json.dump(CFG, f, indent=2)

if not DEPLOYABLE:
    print("!" * 70)
    print("This checkpoint did not pass the release gate. Files are written")
    print("so the run is not lost, but DO NOT copy them into models/.")
    print("!" * 70)

size_mb = os.path.getsize(f"{OUT_DIR}/dermascan_b3.pt") / 1024 / 1024
print(f"dermascan_b3.pt          {size_mb:.0f} MB   fingerprint {fingerprint}")
for f_ in ["class_thresholds.json", "calibration.json", "evaluation_results.json",
           "train_config.json", "training_history.csv", "training_curves.png",
           "confusion_matrix_measured.png",
           "split_train.csv", "split_val.csv", "split_calib.csv", "split_test.csv"]:
    print("   ", f_)

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
This checkpoint did not pass the release gate. Files are written
so the run is not lost, but DO NOT copy them into models/.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
dermascan_b3.pt          41 MB   fingerprint 27629fd06174b38f
    class_thresholds.json
    calibration.json
    evaluation_results.json
    train_config.json
    training_history.csv
    training_curves.png
    confusion_matrix_measured.png
    split_train.csv
    split_val.csv
    split_calib.csv
    split_test.csv


In [18]:
# ── Verify the export loads on CPU, exactly as the server will load it ───────
# This is the check that would have caught the previous deployment's silent
# architecture mismatch. Do not skip it.
ck = torch.load(f"{OUT_DIR}/dermascan_b3.pt", map_location="cpu", weights_only=True)
print("weights_only load: OK (no numpy scalars in the bundle)")

cpu_model = (timm.create_model(ck["arch"], pretrained=False,
                               num_classes=ck["num_classes"])
             if ck["head"] == "plain"
             else MultiHeadNet(ck["arch"], ck["num_classes"], 0.0, 0.0))
cpu_model.load_state_dict(ck["model_state_dict"])   # strict
cpu_model.eval()
print("strict state_dict load on CPU: OK")

probe = torch.randn(1, 3, ck["img_size"], ck["img_size"])
with torch.no_grad():
    out = cpu_model(probe)
assert out.shape == (1, ck["num_classes"]), out.shape
print(f"CPU forward pass: OK  logits {tuple(out.shape)}")
print("classes:", ck["classes"])
print("img_size:", ck["img_size"], "| readout:", ck["readout"],
      "| T:", round(ck["temperature"], 3), "| mel alert:", ck["mel_alert_threshold"])

weights_only load: OK (no numpy scalars in the bundle)
strict state_dict load on CPU: OK
CPU forward pass: OK  logits (1, 7)
classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
img_size: 300 | readout: softmax | T: 1.0 | mel alert: 0.5


## 6b. Collect the output before you lose it

In [19]:
# ── Collect the output, and check it will survive this session ───────────────
# Kaggle deletes /kaggle/working when an unsaved interactive session ends. Run 3
# was lost that way: 50 minutes of T4 time, a model that passed the release gate,
# and no way to get the weights back - the notebook file keeps the printed
# metrics but not the 41 MB of weights they describe.
import hashlib

run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "unknown")
print(f"kernel run type: {run_type}")
if run_type != "Batch":
    print()
    print("!" * 72)
    print("THESE FILES ARE NOT SAVED YET.")
    print("This is an interactive session, so /kaggle/working is deleted when it")
    print("ends - closing the tab is enough, and version history will be empty.")
    print()
    print("Do ONE of these before you go anywhere:")
    print("  a) Save Version -> Quick Save   (keeps this session's outputs as a")
    print("     permanent version; does not re-run anything)")
    print("  b) download the files listed below from the Output panel now")
    print()
    print("For the next run, prefer Save Version -> Save & Run All (Commit): it")
    print("executes top to bottom on Kaggle's side and persists the output whether")
    print("or not your browser is open.")
    print("!" * 72)
    print()

WANTED = ["dermascan_b3.pt", "class_thresholds.json", "calibration.json",
          "evaluation_results.json", "train_config.json",
          "confusion_matrix_measured.png",
          "training_curves.png", "training_history.csv",
          "split_train.csv", "split_val.csv", "split_calib.csv", "split_test.csv"]

print(f"{'file':<34}{'size':>12}  sha256[:16]")
total = 0
for name in WANTED:
    p = f"{OUT_DIR}/{name}"
    if not os.path.exists(p):
        print(f"{name:<34}{'MISSING':>12}")
        continue
    size = os.path.getsize(p)
    total += size
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    unit = f"{size/1024/1024:.1f} MB" if size > 1 << 20 else f"{size/1024:.1f} KB"
    print(f"{name:<34}{unit:>12}  {h.hexdigest()[:16]}")
print(f"{'':<34}{total/1024/1024:>9.1f} MB total")

print()
print("Download all of it, not just the .pt - the thresholds and calibration are")
print("what make the checkpoint servable, and scripts/deploy_checkpoint.py will")
print("refuse to install without them.")
print()
print("  Browser:  Output panel (right sidebar) -> Download All")
print("  CLI:      kaggle kernels output <user>/<notebook-slug> -p ./kaggle_run")
print()
print("Then, in the serving repo:")
print("  python scripts/deploy_checkpoint.py --from-dir ./kaggle_run --dry-run")
print()
print("The sha256 values above are the integrity check: a truncated download")
print("will not match, and dermascan_b3.pt should be ~41 MB.")

kernel run type: Interactive

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
THESE FILES ARE NOT SAVED YET.
This is an interactive session, so /kaggle/working is deleted when it
ends - closing the tab is enough, and version history will be empty.

Do ONE of these before you go anywhere:
  a) Save Version -> Quick Save   (keeps this session's outputs as a
     permanent version; does not re-run anything)
  b) download the files listed below from the Output panel now

For the next run, prefer Save Version -> Save & Run All (Commit): it
executes top to bottom on Kaggle's side and persists the output whether
or not your browser is open.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

file                                      size  sha256[:16]
dermascan_b3.pt                        41.3 MB  8faac52cf5a0f4af
class_thresholds.json                   1.2 KB  55d392fdfcd43921
calibration.json                        0.4 KB  0abf236922516a83
eval

## 7. Deploying into the serving repository

Download everything the cell above listed - the Output panel's **Download All**,
or:

```bash
kaggle kernels output <user>/<notebook-slug> -p ./kaggle_run
```

Then, in `MODEL_Skin-Cancer/`, let the installer do it. Copying by hand is how
the previous deployment ended up serving a different network than the one its
published metrics described:

```bash
python scripts/deploy_checkpoint.py --from-dir ./kaggle_run --dry-run
python scripts/deploy_checkpoint.py --from-dir ./kaggle_run
```

It re-fingerprints the weights against what the bundle claims, loads them
strictly into the architecture `backend/model.py` actually builds, runs a CPU
forward pass, checks `IMG_SIZE` and `MODEL_ARCH` against the bundle, confirms the
threshold and calibration sidecars agree with it, and re-checks the release gate
before copying anything. It backs up the live files first; `--rollback <stamp>`
puts them back.

`calibration.json` carries `readout`, so the backend picks up softmax
automatically - a `READOUT` value in `.env` is only a fallback, and a stale one
there will mislead whoever reads the config next.

Verify before trusting it:

```bash
python -m pytest
python scripts/evaluate_model.py --data-dir path/to/test_set
```

The evaluator's numbers should match `docs/evaluation_results.json`. **If they do
not, the serving path and the training path have diverged again** - that
disagreement is precisely the bug this whole pipeline is built to surface, so
stop and find it rather than publishing either number.

### Honest expectations

Melanoma recall is bounded by how separable melanoma and nevus are at this
resolution with this much data. Class weighting and the alert channel shift the
operating point; they do not add information. If recall is still short of what
you need, the next levers are more melanoma data, higher input resolution, or a
dedicated melanoma-versus-nevus head - not further threshold tuning, which only
moves along the same curve.